### 📊 CodeCritic Provider Log Analysis — Execution Trace & Latency Breakdown

This notebook performs exploratory data analysis (EDA) on the **`provider_log`** table from a CodeCritic execution session. It reconstructs execution chains, visualizes how provider types interact across FSM layers, and surfaces system latency and stability characteristics.

#### 🔍 Overview

* **Execution Timeline**
  Detailed scatter and instance-based visualizations of provider execution across time.

* **Call Frequency**
  Bar chart summarizing how frequently each provider type is invoked.

* **Latency Analysis**

  * Total latency by provider type
  * Average latency per call
  * Detailed treemaps showing hierarchical execution cost

* **Execution Graphs**

  * Directed graph of file-based transformations
  * Sankey diagram of provider interactions
  * FSM-nested treemap reconstruction

### 🛠️ Project Root Setup + Database Reset and Seeding

This cell resets the entire database schema and seeds it with initial data. It is typically used in local development or test environments before running sessions.

#### 🧠 What this code does:

1. **Sets the working directory** to the project root (`C:\Repos\codecritic`) so all relative imports and paths resolve correctly.
2. **Imports all seeders** from `app.db.seeders` to populate the database with provider and configuration records.
3. **Drops and recreates** all tables using SQLAlchemy’s `Base.metadata.drop_all()` and `create_all()` for a clean start.
4. **Runs all seeders** in a single transaction using `Session(bind=engine)` to insert:
   - Prompts and prompt providers
   - Tools, scores, and agent engines
   - Agents, states, systems, controllers, programs, and sessions
5. **Loads environment variables** from `env/.env` relative to the project root.

#### 🔍 Why it matters:

- Ensures the system starts from a **clean and reproducible state**
- Guarantees that all required provider records are seeded before executing sessions
- Makes local development or testing **deterministic**
- Prepares the environment for any downstream scripts, agents, or workflows

Use this cell at the start of notebooks that need access to a freshly initialized and fully seeded CodeCritic environment.


In [ ]:
from pathlib import Path
import shutil
import os, sys
from dotenv import load_dotenv

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

# Define the directories to be emptied
directories_to_clear = [
    PROJECT_ROOT / "working_files",
    PROJECT_ROOT / "extensions",
    PROJECT_ROOT / "experiments" / "snapshots"
]

# Function to delete all files and directories within a specified directory
def clear_directory(directory: Path):
    if directory.exists() and directory.is_dir():
        for item in directory.iterdir():
            try:
                if item.is_dir():
                    shutil.rmtree(item)  # Remove directory and all its contents
                else:
                    item.unlink()  # Remove file
            except Exception as e:
                print(f"Error deleting {item}: {e}")

# Clear all the directories
for directory in directories_to_clear:
    clear_directory(directory)

print("All specified directories have been cleared.")

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_score_providers import seed_score_providers
from app.db.seeders.seed_tool_providers import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.seeders.seed_state_providers import seed_state_providers
from app.db.seeders.seed_system_providers import seed_system_providers
from app.db.seeders.seed_controller_providers import seed_controller_providers
from app.db.seeders.seed_program_provider import seed_program_providers
from app.db.seeders.seed_session_configs import seed_session_configs
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)
    seed_agent_providers(session)
    seed_state_providers(session)
    seed_system_providers(session)
    seed_controller_providers(session)
    seed_program_providers(session)
    seed_session_configs(session)

# Load from env/.env relative to project root
dotenv_path = Path("env/.env").resolve()
if dotenv_path.exists():
    load_dotenv(dotenv_path)
    print("✅ Loaded environment variables from env/.env")
else:
    raise FileNotFoundError(f"❌ Missing .env file at: {dotenv_path}")


### 🧪 Run Program Using SessionConfig

This cell loads and executes a **program** using configuration data from the database, simulating a specific session's execution environment.

#### 🧠 What this code does:

* **Fetches the session configuration** from the database (with `session_id = 1`), ensuring the configuration is available.
* Prepares a **test file** (`session_linked_execution.py`) with a basic function to run.
* Constructs input data for the program, including the file path, session ID, and system context (`LINTING`).
* **Runs the program** using a factory method (`ProgramProviderFactory`), passing the input data along with session-specific configuration.
* Prints the **final result** from the program, showing the output of the execution.

#### 🔍 What you're observing:

* This step demonstrates how the system initializes the **program provider** and runs it in the context of a **specific session**.
* It’s useful for **debugging execution flows** and **validating program behavior** based on dynamic session configurations.
* **Outputs** include any results generated by the program, including runtime metadata and system responses.

This approach is fundamental for simulating **real-time program execution** tied to specific session states.


In [ ]:
# 🧪 Run Program Using SessionConfig

from app.db.models import SessionConfig
from app.enums.logging_enums import PROVIDER_TYPE
from app.enums.system_enums import SYSTEM
from app.factories.program_provider_factory import ProgramProviderFactory
from sqlalchemy.orm import Session
from pathlib import Path
import json

# Fetch session config
with Session(bind=engine) as db:
    session_row = db.query(SessionConfig).filter_by(id=1).first()
    assert session_row, "❌ No session config found"

# Prepare test file
file = Path("tests/complex_lint_task.py")

# Construct input using session metadata
input_data = {
    "file_path":  str(file),
    "session_id": session_row.id,
    "system":     SYSTEM.LINTING.value
}

# Run associated program
program = ProgramProviderFactory.create(
    id=session_row.program_provider_id,
    called_by_type=PROVIDER_TYPE.SESSION,
    called_by_id=session_row.id
)
result = program.run(input_data, session_id=session_row.id)

print("✅ Final Program Result (via SessionConfig):")
print(json.dumps(result.model_dump(), indent=2))


### 🧾 Full Execution Trace (Provider Log Step-by-Step)

This code cell loads and prints a **chronological step-by-step trace** of provider executions for a specific session.

#### 🧠 What this code does:
- Connects to the SQLite database and loads `provider_log` rows for `session_id = 1`.
- Sorts the entries by timestamp to reconstruct the actual execution timeline.
- Iterates over each row and prints a formatted log containing:
  - Timestamp and provider metadata
  - File name and latency
  - Config and call chain info
  - Parsed input and output fields (truncated if long)

#### 🔍 What you're observing:
- Each **step** shows the complete context of a provider call:
  - Who it was **called by**
  - What it **received** (input)
  - What it **produced** (output)
- Provides an **audit-like view** that’s useful for debugging, compliance, or understanding system orchestration logic.
- Can reveal:
  - Repeated calls
  - Delays or outliers
  - Input/output mismatches
  - Provider chain inconsistencies

This is the foundation for building trace visualizations, debugging sessions, or generating compliance audit trails.


In [ ]:
import sqlite3
import pandas as pd
import json
from app.db.connection import DB_PATH

# Connect and load logs
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM provider_log WHERE session_id='1'", conn)
conn.close()

# Sort by timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(by='timestamp').reset_index(drop=True)

# Display trace
for i, row in df.iterrows():
    print(f"\n🔹 Step {i + 1}")
    print(f"🕒 {row['timestamp']}")
    print(f"🧩 Provider: {row['provider_type']} (ID {row['provider_id']})")
    print(f"📦 Schema: {row['output_schema']}")
    print(f"⏱️  {row['latency_ms']} ms  |  📄 {row['file_name']}")
    print(f"🔑 Config Hash: {row['config_hash']}")

    # 🧭 Call chain
    caller = f"{row['called_by_type']} (ID {row['called_by_id']})" if row['called_by_type'] or row['called_by_id'] else "None"
    print(f"🧭 Called by: {caller}")

    # 📥 Input
    try:
        parsed_input = json.loads(row["input"])
        short_input = json.dumps(parsed_input, indent=2)
        print("📥 Input:")
        print(short_input if len(short_input) < 500 else short_input[:500] + "...\n  (truncated)")
    except Exception:
        print("📥 Input (raw):")
        print(row["input"])

    # 📤 Output
    try:
        parsed_output = json.loads(row["output"])
        short_output = json.dumps(parsed_output, indent=2)
        print("📤 Output:")
        print(short_output if len(short_output) < 500 else short_output[:500] + "...\n  (truncated)")
    except Exception:
        print("📤 Output (raw):")
        print(row["output"])

    print("—" * 80)


### 📦 Treemap of Provider Execution Cost

This chart presents a **latency-weighted treemap** that reveals the **nested execution structure** of a CodeCritic run — from top-level orchestration down to fine-grained tooling.

#### 🧠 What this code does:
- **Builds a call hierarchy** by tracing each provider’s `called_by_type`, recursively reconstructing the execution path for every log entry.
- **Flattens each path** into separate columns (e.g., `path_0`, `path_1`, ...) so Plotly can visualize the full call stack as a nested tree.
- **Visualizes latency cost** using a treemap, where:
  - Each **box** represents a provider type at a specific nesting level.
  - **Box size** is proportional to total `latency_ms` contributed by that node and its descendants.
  - **Color** corresponds to `provider_type`, using a colorblind-safe palette.

#### 🔍 What you're observing:
- The **upper-left boxes** show top-level orchestrators (e.g., `session`, `program`).
- As you move inward (right and down), you see **nested calls** to `controller`, `system`, `state`, and ultimately `agent`, `tool`, `score`, etc.
- **Larger boxes** deeper in the tree indicate subsystems that consume significant time (e.g., tools like `mypy`, `black`, `ruff`).
- Hovering on a box reveals its exact path and latency.

This visualization is ideal for:
- Identifying where latency accumulates in multi-step workflows.
- Spotting repeated or redundant calls.
- Auditing nested delegation patterns during FSM-driven execution.


In [ ]:
import plotly.express as px
import pandas as pd

# Non-destructive copy
df_treemap = df.copy()

# Truncate provider_type
df_treemap["provider_label"] = df_treemap["provider_type"].astype(str).str.split(".").str[-1]
df_treemap["source"] = df_treemap["called_by_type"].fillna("root")
df_treemap["target"] = df_treemap["provider_label"]  # simplified target

# Edge list from simplified names
edges = list(zip(df_treemap["source"], df_treemap["target"]))

# Recursive path builder
def build_path(row, edges):
    if row["source"] == "root":
        return ["root", row["target"]]
    for src, tgt in edges:
        if tgt == row["source"]:
            return build_path({"source": src, "target": tgt}, edges) + [row["target"]]
    return ["root", row["source"], row["target"]]

# Build full hierarchical path
df_treemap["call_path"] = df_treemap.apply(lambda row: build_path(row, edges), axis=1)

# Flatten and clean hierarchy
path_df = df_treemap["call_path"].apply(pd.Series)
path_df.columns = [f"path_{i}" for i in path_df.columns]

# Add final disambiguator for leaf nodes
path_df[path_df.columns[-1]] = df_treemap.apply(
    lambda row: f"{row['provider_label']} (ID {row['provider_id']})", axis=1
)

# Ensure numeric latency
df_treemap["latency_ms"] = pd.to_numeric(df_treemap["latency_ms"], errors="coerce").fillna(0)

# Combine everything
sunburst_data = pd.concat([df_treemap, path_df], axis=1)

# Plot
fig = px.treemap(
    sunburst_data,
    path=path_df.columns.tolist(),
    values="latency_ms",
    color="provider_label",
    color_discrete_sequence=px.colors.qualitative.Safe,
    title="📦 Treemap of Provider Execution Cost"
)

fig.update_traces(root_color="lightblue")
fig.show()


### 🧩 Execution Flow (Provider Type Nesting)

This Sankey diagram visualizes the hierarchical execution path taken by the system during a program run.

#### 🔍 What this code does:
- The code groups log entries (`provider_log`) by their `called_by_type` (who invoked them) and `provider_type` (what kind of provider was called).
- It then builds a flow network showing how control moves from one provider type to the next.
- The width of each band represents the number of calls made along that path.

#### 🧠 What you're seeing in the chart:
- **Left-to-right flow** from high-level providers like `session`, `program`, and `controller`, down to mid-level like `system` and `state`, and finally to low-level workers like `agent`, `score`, and `tool`.
- **Wider bands** indicate more frequent interactions. For example, the `state → score` and `score → tool` flows are thick, indicating heavy evaluation and tool usage.
- **Branching paths** (e.g., from `agent` to `score`, `tool`, `agent_engine`) reveal the dependencies that agents rely on to make decisions.

This diagram is useful for identifying:
- Execution bottlenecks
- Overused subsystems
- System architecture layering

It’s a high-level snapshot of how components orchestrate a complete run.


In [ ]:
import plotly.graph_objects as go

# Step 1: Clean copy
df_sankey = df.copy()

# Step 2: Simplify labels to suffix
df_sankey["source"] = df_sankey["called_by_type"].fillna("start")
df_sankey["target"] = df_sankey["provider_type"]
df_sankey["source_label"] = df_sankey["source"].apply(lambda x: str(x).split(".")[-1])
df_sankey["target_label"] = df_sankey["target"].apply(lambda x: str(x).split(".")[-1])

# Step 3: Enforce node order by appearance in time
ordered_labels = pd.concat([df_sankey["source_label"], df_sankey["target_label"]], ignore_index=True).dropna().unique().tolist()
node_indices = {label: i for i, label in enumerate(ordered_labels)}

# Step 4: Build link data
link_data = (
    df_sankey.groupby(["source_label", "target_label"])
    .size()
    .reset_index(name="count")
    .assign(
        source_idx=lambda d: d["source_label"].map(node_indices),
        target_idx=lambda d: d["target_label"].map(node_indices)
    )
)

# Step 5: Plot Sankey
fig = go.Figure(go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=ordered_labels
    ),
    link=dict(
        source=link_data["source_idx"],
        target=link_data["target_idx"],
        value=link_data["count"]
    )
))

fig.update_layout(title_text="🧩 Execution Flow", font_size=14)
fig.show()


### 🕒 Timeline of Provider Execution

This scatter plot displays a time-ordered trace of all provider invocations that occurred during the session.

#### 🧠 What this code does:
- Uses `plotly.express.scatter` to plot each row from `provider_log` based on its `timestamp` and `provider_type`.
- Each dot represents a single provider call.
- Hovering over a point reveals metadata: `provider_id`, `called_by_type`, `called_by_id`, and `latency_ms`.

#### 🔍 What you're observing:
- Execution begins at the top (`program`) and flows down through `controller`, `system`, `state`, and into `agent`, `score`, and `tool`.
- The **dense lower bands** (especially `tool` and `score`) indicate high-frequency, low-latency invocations.
- **Temporal clustering** shows how providers group during nested FSM transitions.
- You can use this view to:
  - Spot bottlenecks
  - Identify execution gaps
  - Debug step timing issues
  - Correlate structural vs runtime behavior

This chart complements the **Sankey diagram** by emphasizing *when* events happened rather than *how* they are structurally connected.


In [ ]:
import plotly.express as px

# Step 1: Copy and create unique instance identifier
df_timeline = df.copy()
df_timeline["provider_label"] = df_timeline["provider_type"].astype(str).str.split(".").str[-1]
df_timeline["provider_instance"] = df_timeline.apply(
    lambda row: f"{row['provider_label']} (ID {row['provider_id']})", axis=1
)

# Step 2: Get vertical order (preserve call order)
instance_order = df_timeline.sort_values("timestamp")["provider_instance"].unique().tolist()

# Step 3: Plot
fig = px.scatter(
    df_timeline,
    x="timestamp",
    y=pd.Categorical(df_timeline["provider_instance"], categories=instance_order, ordered=True),
    color="provider_label",
    hover_data=["provider_id", "called_by_type", "called_by_id", "latency_ms"],
    title="🕒 Timeline of Provider Execution (Detailed)"
)

# Step 4: Visual tuning
fig.update_traces(marker=dict(size=12, line=dict(width=1, color="DarkSlateGrey")))
fig.update_layout(
    yaxis_title="Provider Instance",
    xaxis_title="Timestamp",
    legend_title="Provider Type",
    height=800
)

fig.show()



### 📊 Frequency of Provider Types in Execution

This bar chart displays the **number of times each `provider_type` appears** in the execution log.

#### 🧠 What this code does:
- Groups the log (`df`) by `provider_type` and counts how many entries exist for each type (e.g., `agent`, `tool`, `score`, `state`, etc.).
- Uses Plotly Express to render a **bar chart** with the provider types on the x-axis and their respective frequencies on the y-axis.
- Adds labels and color-coding to make the chart more interpretable and visually distinct.

#### 🔍 What you're observing:
- **Higher bars** indicate providers that are invoked more frequently (e.g., tools may appear often during scoring).
- Can help reveal which components dominate system activity.
- Serves as a quick summary of system behavior — useful for tuning, bottleneck detection, or debugging repeated execution patterns.

This is a foundational metric for system observability, especially when comparing runs or systems with different configurations.


In [ ]:
import plotly.express as px

# Create a copy to avoid mutating the original DataFrame
df_bar = df.copy()

# Truncate provider_type to the part after the last '.'
df_bar['provider_type'] = df_bar['provider_type'].apply(lambda x: x.split('.')[-1])

# Count how many times each provider type appears
provider_freq = df_bar["provider_type"].value_counts().reset_index()
provider_freq.columns = ["provider_type", "count"]

# Plot as bar chart
fig = px.bar(
    provider_freq,
    x="provider_type",
    y="count",
    text="count",
    title="📊 Frequency of Provider Types in Execution",
    labels={"provider_type": "Provider Type", "count": "Occurrences"},
    color="provider_type",
    color_discrete_sequence=px.colors.qualitative.Set3
)

# Update trace text position and layout
fig.update_traces(textposition="outside")

# Increase height and adjust margins for better space allocation
fig.update_layout(
    height=600,
    margin=dict(t=100, b=100, l=50, r=50),
    xaxis_title="Provider Type", 
    yaxis_title="Count"
)

fig.show()


### ⏱️ Total Latency by Provider Type

This bar chart summarizes the **cumulative execution time** consumed by each `provider_type` across the entire session.

#### 🧠 What this code does:
- Aggregates the `latency_ms` field for each `provider_type` using `.groupby()` and `.sum()`.
- Sorts the result so the most time-consuming providers appear first.
- Uses Plotly Express to render a labeled **bar chart** with:
  - Provider types on the x-axis
  - Total latency (in milliseconds) on the y-axis
- Applies a pastel color palette for clarity and visual grouping.

#### 🔍 What you're observing:
- This reveals which components are the **most performance-intensive** — e.g., `agent_engine`, `tool`, or `system`.
- It complements frequency charts by showing not just how often a provider is used, but how **expensive** it is.
- Useful for identifying where to focus **optimization**, **caching**, or **parallelization** efforts.

This chart is essential for profiling execution cost at a subsystem level and for balancing responsiveness across an FSM-managed system.


In [ ]:
import plotly.express as px

# Work on a copy to avoid modifying the original DataFrame
df_latency = df.copy()

# Truncate provider_type to the part after the last '.'
df_latency['provider_type'] = df_latency['provider_type'].apply(lambda x: x.split('.')[-1])

# Sum latency by provider type
latency_by_type = (
    df_latency.groupby("provider_type")["latency_ms"]
    .sum()
    .reset_index()
    .sort_values("latency_ms", ascending=False)
)

# Plot as bar chart
fig = px.bar(
    latency_by_type,
    x="provider_type",
    y="latency_ms",
    text="latency_ms",
    title="⏱️ Total Latency by Provider Type",
    labels={"provider_type": "Provider Type", "latency_ms": "Total Latency (ms)"},
    color="provider_type",
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# Adjust text position and layout
fig.update_traces(textposition="outside")

# Improve layout for better readability
fig.update_layout(
    height=600,
    xaxis_title="Provider Type", 
    yaxis_title="Total Latency (ms)",
    margin=dict(l=50, r=50, t=50, b=100)
)

# Show the plot
fig.show()


### ⏱️ Average Latency per Call by Provider Type

This bar chart displays the **average execution time per invocation** for each `provider_type`, providing a normalized view of performance cost.

#### 🧠 What this code does:
- Groups the log (`df`) by `provider_type`.
- Calculates:
  - **Total latency** (`sum`)
  - **Call count** (`count`)
  - **Average latency per call** (normalized by dividing total by count)
- Rounds the average values to **integers** for cleaner chart labels.
- Renders a **bar chart** with color-coded provider types using a consistent palette.

#### 🔍 What you're observing:
- This view accounts for how frequently a provider is called, exposing **per-call cost**.
- Helpful for spotting **low-frequency but expensive providers** (e.g., `agent_engine`) vs. **high-frequency lightweight tools**.
- Guides decisions around **caching**, **parallelization**, or **refactoring** based on execution efficiency.

Use this alongside total latency charts to balance **frequency** and **impact** across the system.


In [ ]:
import plotly.express as px

# Work on a copy to preserve the original DataFrame
df_avg_latency = df.copy()

# Truncate provider_type to the part after the last '.'
df_avg_latency['provider_type'] = df_avg_latency['provider_type'].apply(lambda x: x.split('.')[-1])

# Compute total latency and call count per provider type
latency_stats = (
    df_avg_latency.groupby("provider_type")["latency_ms"]
    .agg(total_latency="sum", call_count="count")
    .reset_index()
)

# Calculate average latency per call
latency_stats["avg_latency_per_call"] = latency_stats["total_latency"] / latency_stats["call_count"]

# Convert for display
latency_stats["avg_latency_label"] = latency_stats["avg_latency_per_call"].round().astype(int)

# Sort for better visual layout
latency_stats = latency_stats.sort_values("avg_latency_per_call", ascending=False)

# Plot
fig = px.bar(
    latency_stats,
    x="provider_type",
    y="avg_latency_per_call",
    text="avg_latency_label",
    title="⏱️ Average Latency per Call by Provider Type",
    labels={"provider_type": "Provider Type", "avg_latency_per_call": "Avg Latency (ms)"},
    color="provider_type",
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(textposition="outside")

fig.update_layout(
    height=600,
    margin=dict(t=100, b=100, l=50, r=50),
    xaxis_title="Provider Type",
    yaxis_title="Average Latency (ms)"
)

fig.show()
